In [ ]:
import cv2
from ultralytics import YOLO

# Load YOLOv8s pretrained model (more accurate than yolov8n)
model = YOLO("yolov8s.pt")

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam")
    exit()

print("Starting webcam... Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    # Run YOLO detection on the frame
    results = model(frame, verbose=False)  # suppress YOLO console logs

    # Keep track of detected objects to avoid repeated prints
    detected_this_frame = []

    # Loop through detected boxes
    for box, cls, conf in zip(results[0].boxes.xyxy, results[0].boxes.cls, results[0].boxes.conf):
        if float(conf) < 0.5:  # skip low-confidence predictions
            continue
            q
        class_name = results[0].names[int(cls)]

        # Draw rectangle around detected object
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Put class name and confidence above rectangle
        label = f"{class_name} {float(conf)*100:.1f}%"
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # Print object name only once per frame
        if class_name not in detected_this_frame:
            print(f"Detected: {class_name}")
            detected_this_frame.append(class_name)

    # Display the annotated frame
    cv2.imshow("Object Detection", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release webcam and close window
cap.release()
cv2.destroyAllWindows()
print("Webcam closed.")